In [11]:
import os
import glob
import pandas as pd
from pathlib import Path

# 1. Find where this notebook is saved
try:
    current_dir = Path(__file__).resolve().parent
except NameError:
    current_dir = Path(os.getcwd()).resolve()

# 2. Locate FinalProject whether the kernel starts in notebooks, FinalProject,
# or the top-level cosmos workspace.
project_candidates = [
    current_dir.parent if current_dir.name == 'notebooks' else current_dir,
    current_dir / '26-the-deep-learners-analysis' / 'FinalProject',
]
project_root = next(
    (path for path in project_candidates if (path / 'data' / 'raw').exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate FinalProject/data/raw from the current directory.')

input_folder = project_root / "data" / "interim"
output_folder = project_root / "data" / "processed"
output_filename = "merged_features.csv"

# Create the processed folder automatically if it's missing
os.makedirs(output_folder, exist_ok=True)

# 3. Grab all CSV files inside that precise folder
search_path = os.path.join(input_folder, "*.csv")
csv_files = sorted(glob.glob(search_path))
csv_files = [f for f in csv_files if os.path.basename(f) != output_filename]
base_file = str(input_folder / "additional_pair_features.csv")
if base_file not in csv_files:
    raise FileNotFoundError(
        f"Required base file not found: {base_file}"
    )
merge_files = [f for f in csv_files if f != base_file]

print(f"Project root directory: {project_root}")
print(f"Looking inside input path: {input_folder}")
print(f"Successfully found {len(csv_files)} file(s) to merge!")


Project root directory: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject
Looking inside input path: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject\data\interim
Successfully found 10 file(s) to merge!


In [12]:
# Use the canonical additional-pair table as the row universe.
# This prevents outer joins from creating new rows whose combined
# daily-text feature is undefined.
print(f"Reading base file: {base_file}")
master_df = pd.read_csv(base_file)
master_df.columns = master_df.columns.str.strip().str.lower()
master_df = master_df.loc[:, ~master_df.columns.str.startswith("unnamed:")]

id_names = ["user_a", "user_b"]
missing_base_keys = [key for key in id_names if key not in master_df.columns]
if missing_base_keys:
    raise KeyError(f"Base file is missing pair keys: {missing_base_keys}")
if master_df.duplicated(id_names).any():
    raise ValueError("Base file contains duplicate participant pairs")
print(f"Tracking keys detected: {id_names}")



Reading base file: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject\data\interim\additional_pair_features.csv
Tracking keys detected: ['user_a', 'user_b']


In [13]:
# 3. Loop through and merge the remaining files sideways
for file in merge_files:
    print(f"Merging: {file}")
    df_next = pd.read_csv(file)
    
    # Clean up column names to prevent case or space mismatches
    df_next.columns = df_next.columns.str.strip().str.lower()
    df_next = df_next.loc[:, ~df_next.columns.str.startswith("unnamed:")]
    
    # Double check if both id_names are present in this specific file
    missing_keys = [key for key in id_names if key not in df_next.columns]
    if missing_keys:
        print(f"⚠️ Warning: Skipping {file} because it is missing tracking key columns: {missing_keys}")
        print(f"   Available columns in this file: {list(df_next.columns)}")
        continue  # Safely skip non-pair-level files

    if df_next.duplicated(id_names).any():
        raise ValueError(f"Duplicate participant pairs found in {file}")

    duplicate_features = [
        column for column in df_next.columns
        if column in master_df.columns and column not in id_names
    ]
    if duplicate_features:
        print(f"   Skipping duplicate columns: {duplicate_features}")
        df_next = df_next.drop(columns=duplicate_features)

    master_df = pd.merge(
        master_df,
        df_next,
        on=id_names,
        how="left",
        validate="one_to_one",
    )

# Rebuild the combined daily inbound-text feature from its two inputs.
# In the source feature definition, no recorded inbound texts is 0.
daily_text_columns = [
    "user_a_daily_texts_received",
    "user_b_daily_texts_received",
]
missing_daily_columns = [
    column for column in daily_text_columns
    if column not in master_df.columns
]
if missing_daily_columns:
    raise KeyError(
        f"Cannot rebuild combined daily texts; missing {missing_daily_columns}"
    )

combined_column = "combined_daily_texts_received"
missing_before = (
    int(master_df[combined_column].isna().sum())
    if combined_column in master_df.columns else len(master_df)
)
master_df[daily_text_columns] = master_df[daily_text_columns].fillna(0)
master_df[combined_column] = master_df[daily_text_columns].sum(axis=1)
assert master_df[combined_column].notna().all()

# Requested pair-level features are calculated below before the final CSV is saved.


Merging: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject\data\interim\call_pair_days.csv
Merging: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject\data\interim\call_pair_totals.csv
Merging: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject\data\interim\pair_facebook_friend_counts.csv
Merging: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject\data\interim\pair_longest_call_streaks.csv
Merging: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject\data\interim\pair_longest_text_streaks.csv
Merging: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject\data\interim\pair_total_texts_sent.csv
Merging: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject\data\interim\pairwise_features.csv
Merging: C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysi

In [14]:
import numpy as np
from itertools import combinations

SECONDS_PER_DAY = 24 * 60 * 60
pair_keys = ['user_a', 'user_b']
raw_folder = project_root / 'data' / 'raw'

# Read the raw interaction logs. Some CNS files prefix the first header with '#'.
calls = pd.read_csv(raw_folder / 'calls.csv')
sms = pd.read_csv(raw_folder / 'sms.csv')
bluetooth = pd.read_csv(raw_folder / 'bt_symmetric.csv')
facebook = pd.read_csv(raw_folder / 'fb_friends.csv')
for frame in (calls, sms, bluetooth, facebook):
    frame.columns = frame.columns.str.strip().str.lstrip('#').str.strip().str.lower()

# Canonicalize the already-merged keys so every unordered pair is user_a < user_b.
master_df[pair_keys] = master_df[pair_keys].apply(pd.to_numeric, errors='coerce')
master_df = master_df.dropna(subset=pair_keys)
master_df[['user_a', 'user_b']] = np.sort(master_df[pair_keys].astype('int64'), axis=1)
master_df = master_df.loc[master_df['user_a'].ne(master_df['user_b'])]

# If an interim table contains repeated pair rows, retain the first non-null value
# in each feature column instead of multiplying rows during later joins.
master_df = master_df.groupby(pair_keys, as_index=False, sort=False).first()

# Build the complete pair universe from every valid participant seen in the
# merged features, call logs, or SMS logs. Bluetooth user_b values can include
# non-participant device IDs, so Bluetooth is intentionally not a user roster.
participant_series = [
    master_df['user_a'], master_df['user_b'],
    calls['caller'], calls['callee'],
    sms['sender'], sms['recipient'],
]
all_users = (
    pd.to_numeric(pd.concat(participant_series, ignore_index=True), errors='coerce')
    .dropna().astype('int64')
)
all_users = sorted(all_users.loc[all_users.ge(0)].unique())
all_pairs = pd.DataFrame(combinations(all_users, 2), columns=pair_keys)
master_df = all_pairs.merge(master_df, on=pair_keys, how='left', validate='one_to_one')

# Recalculate Facebook features for the complete pair universe. A participant
# absent from the friendship graph has no recorded connections; this does not
# establish whether that participant had a Facebook account.
facebook_edges = facebook[pair_keys].apply(pd.to_numeric, errors='coerce').dropna()
facebook_edges = facebook_edges.astype('int64')
facebook_edges = facebook_edges.loc[
    facebook_edges['user_a'].ge(0)
    & facebook_edges['user_b'].ge(0)
    & facebook_edges['user_a'].ne(facebook_edges['user_b'])
]
facebook_neighbors = pd.concat(
    [
        facebook_edges.rename(columns={'user_a': 'user', 'user_b': 'friend'}),
        facebook_edges.rename(columns={'user_b': 'user', 'user_a': 'friend'}),
    ],
    ignore_index=True,
).drop_duplicates()
facebook_friend_counts = facebook_neighbors.groupby('user')['friend'].nunique()

user_a_has_facebook_data = master_df['user_a'].isin(facebook_friend_counts.index)
user_b_has_facebook_data = master_df['user_b'].isin(facebook_friend_counts.index)
facebook_data_categories = [
    'both_have_recorded_connections',
    'a_only_has_recorded_connections',
    'b_only_has_recorded_connections',
    'neither_has_recorded_connections',
]
master_df['facebook_data_category'] = pd.Categorical(
    np.select(
        [
            user_a_has_facebook_data & user_b_has_facebook_data,
            user_a_has_facebook_data & ~user_b_has_facebook_data,
            ~user_a_has_facebook_data & user_b_has_facebook_data,
        ],
        facebook_data_categories[:3],
        default='neither_has_recorded_connections',
    ),
    categories=facebook_data_categories,
)

master_df['total_facebook_friend_count'] = (
    master_df['user_a'].map(facebook_friend_counts).fillna(0)
    + master_df['user_b'].map(facebook_friend_counts).fillna(0)
).astype('int64')
facebook_count_categories = [
    'no_recorded_friends',
    'low',
    'moderate',
    'high',
    'very_high',
]
facebook_friend_count = master_df['total_facebook_friend_count']
master_df['total_facebook_friend_count_category'] = pd.Categorical(
    np.select(
        [
            facebook_friend_count.eq(0),
            facebook_friend_count.between(1, 20),
            facebook_friend_count.between(21, 45),
            facebook_friend_count.between(46, 80),
            facebook_friend_count.gt(80),
        ],
        facebook_count_categories,
        default='no_recorded_friends',
    ),
    categories=facebook_count_categories,
    ordered=True,
)
assert master_df[
    [
        'total_facebook_friend_count',
        'facebook_data_category',
        'total_facebook_friend_count_category',
    ]
].notna().all().all()

def canonical_interactions(frame, left_user, right_user):
    """Return timestamped, valid, unordered participant-pair interactions."""
    events = frame[['timestamp', left_user, right_user]].rename(
        columns={left_user: 'left_user', right_user: 'right_user'}
    ).copy()
    for column in events.columns:
        events[column] = pd.to_numeric(events[column], errors='coerce')
    events = events.dropna().astype('int64')
    events = events.loc[
        events['left_user'].ge(0)
        & events['right_user'].ge(0)
        & events['left_user'].ne(events['right_user'])
        & events['left_user'].isin(all_users)
        & events['right_user'].isin(all_users)
    ]
    events['user_a'] = events[['left_user', 'right_user']].min(axis=1)
    events['user_b'] = events[['left_user', 'right_user']].max(axis=1)
    return events[['timestamp', 'user_a', 'user_b']]

# Categorize weekend-vs-weekday activity between the two users. Use distinct
# active pair-days so frequent Bluetooth scans do not swamp calls and texts.
interaction_events = pd.concat(
    [
        canonical_interactions(calls, 'caller', 'callee'),
        canonical_interactions(sms, 'sender', 'recipient'),
        canonical_interactions(bluetooth, 'user_a', 'user_b'),
    ],
    ignore_index=True,
)
interaction_events['elapsed_day'] = interaction_events['timestamp'] // SECONDS_PER_DAY
active_pair_days = interaction_events[pair_keys + ['elapsed_day']].drop_duplicates()
active_pair_days['is_weekend'] = active_pair_days['elapsed_day'].mod(7).isin([0, 6])

week_part = (
    active_pair_days.groupby(pair_keys)['is_weekend']
    .agg(weekend_interaction_days='sum', total_interaction_days='size')
    .reset_index()
)
week_part['weekday_interaction_days'] = (
    week_part['total_interaction_days'] - week_part['weekend_interaction_days']
)
interaction_categories = [
    'no_interaction',
    'weekend_only',
    'weekday_only',
    'both_weekday_dominant',
    'both_equal',
    'both_weekend_dominant',
]
week_part['weekend_weekday_interaction_category'] = np.select(
    [
        week_part['weekday_interaction_days'].eq(0),
        week_part['weekend_interaction_days'].eq(0),
        week_part['weekday_interaction_days'].gt(week_part['weekend_interaction_days']),
        week_part['weekday_interaction_days'].eq(week_part['weekend_interaction_days']),
        week_part['weekend_interaction_days'].gt(week_part['weekday_interaction_days']),
    ],
    interaction_categories[1:],
    default='no_interaction',
)
category_column = 'weekend_weekday_interaction_category'
master_df = master_df.drop(
    columns=['weekend_weekday_interaction_ratio', category_column],
    errors='ignore',
)
master_df = master_df.merge(
    week_part[pair_keys + [category_column]],
    on=pair_keys,
    how='left',
    validate='one_to_one',
)
master_df[category_column] = pd.Categorical(
    master_df[category_column].fillna('no_interaction'),
    categories=interaction_categories,
)
assert master_df[category_column].notna().all()

# Preserve both the direction and degree of imbalance in pair communication.
reciprocity_categories = [
    'no_interaction',
    'a_only',
    'b_only',
    'a_much_more',
    'a_more',
    'balanced',
    'b_more',
    'b_much_more',
]

def build_directional_reciprocity_category(
    frame, sender_column, recipient_column, category_name
):
    """Categorize the direction and balance of communication for each pair."""
    directed = frame[[sender_column, recipient_column]].rename(
        columns={sender_column: 'sender', recipient_column: 'recipient'}
    ).copy()
    directed[['sender', 'recipient']] = directed[['sender', 'recipient']].apply(
        pd.to_numeric, errors='coerce'
    )
    directed = directed.dropna().astype('int64')
    directed = directed.loc[
        directed['sender'].ge(0)
        & directed['recipient'].ge(0)
        & directed['sender'].ne(directed['recipient'])
        & directed['sender'].isin(all_users)
        & directed['recipient'].isin(all_users)
    ].copy()
    directed['user_a'] = directed[['sender', 'recipient']].min(axis=1)
    directed['user_b'] = directed[['sender', 'recipient']].max(axis=1)
    directed['from_a'] = directed['sender'].eq(directed['user_a'])

    direction_counts = (
        directed.groupby(pair_keys + ['from_a'])
        .size()
        .unstack(fill_value=0)
    )
    from_a = direction_counts.get(True, pd.Series(0, index=direction_counts.index))
    from_b = direction_counts.get(False, pd.Series(0, index=direction_counts.index))
    larger_count = pd.concat([from_a, from_b], axis=1).max(axis=1)
    balance_ratio = from_a.combine(from_b, min).div(larger_count)

    category = np.select(
        [
            from_a.gt(0) & from_b.eq(0),
            from_b.gt(0) & from_a.eq(0),
            balance_ratio.ge(0.8),
            from_a.gt(from_b) & balance_ratio.ge(0.5),
            from_a.gt(from_b) & balance_ratio.lt(0.5),
            from_b.gt(from_a) & balance_ratio.ge(0.5),
            from_b.gt(from_a) & balance_ratio.lt(0.5),
        ],
        [
            'a_only',
            'b_only',
            'balanced',
            'a_more',
            'a_much_more',
            'b_more',
            'b_much_more',
        ],
        default='no_interaction',
    )
    return pd.DataFrame(
        {
            'user_a': direction_counts.index.get_level_values('user_a'),
            'user_b': direction_counts.index.get_level_values('user_b'),
            category_name: category,
        }
    )

reciprocity_tables = [
    build_directional_reciprocity_category(
        sms, 'sender', 'recipient', 'text_reciprocity_category'
    ),
    build_directional_reciprocity_category(
        calls, 'caller', 'callee', 'call_reciprocity_category'
    ),
]
reciprocity_category_columns = [
    'text_reciprocity_category',
    'call_reciprocity_category',
]
master_df = master_df.drop(
    columns=[
        'text_reciprocity',
        'call_reciprocity',
        *reciprocity_category_columns,
    ],
    errors='ignore',
)
for category_name, category_table in zip(
    reciprocity_category_columns, reciprocity_tables
):
    master_df = master_df.merge(
        category_table,
        on=pair_keys,
        how='left',
        validate='one_to_one',
    )
    master_df[category_name] = pd.Categorical(
        master_df[category_name].fillna('no_interaction'),
        categories=reciprocity_categories,
    )
    assert master_df[category_name].notna().all()

# Convert completed-call durations into consistent, interpretable categories.
call_duration_categories = [
    'no_completed_call',
    'zero_duration_call',
    'very_short',
    'short',
    'medium',
    'long',
    'very_long',
]
call_duration_columns = {
    'minimum_call_contact_time_seconds': 'minimum_call_contact_time_category',
    'average_call_contact_time': 'average_call_contact_time_category',
    'max_call_contact_time': 'max_call_contact_time_category',
}

def categorize_call_duration(duration):
    """Categorize a call duration measured in seconds."""
    numeric_duration = pd.to_numeric(duration, errors='coerce')
    if numeric_duration.dropna().lt(0).any():
        raise ValueError('Completed-call duration columns cannot contain negative values')

    labels = np.select(
        [
            numeric_duration.isna(),
            numeric_duration.eq(0),
            numeric_duration.gt(0) & numeric_duration.le(30),
            numeric_duration.gt(30) & numeric_duration.le(60),
            numeric_duration.gt(60) & numeric_duration.le(180),
            numeric_duration.gt(180) & numeric_duration.le(600),
            numeric_duration.gt(600),
        ],
        call_duration_categories,
        default='no_completed_call',
    )
    return pd.Categorical(labels, categories=call_duration_categories, ordered=True)

missing_duration_columns = [
    column for column in call_duration_columns if column not in master_df.columns
]
if missing_duration_columns:
    raise KeyError(
        f'Cannot categorize call durations; columns not found: {missing_duration_columns}'
    )

for source_column, category_name in call_duration_columns.items():
    master_df[category_name] = categorize_call_duration(master_df[source_column])

call_duration_category_columns = list(call_duration_columns.values())
master_df = master_df.drop(columns=list(call_duration_columns))
assert master_df[call_duration_category_columns].notna().all().all()

# Categorize average Bluetooth signal strength. Missing means that the pair had
# no proximity measurement; it must not be treated as an RSSI value of zero.
proximity_signal_categories = [
    'no_measurement',
    'very_weak',
    'weak',
    'moderate',
    'strong',
    'very_strong',
]
average_rssi = pd.to_numeric(master_df['average_proximity_rssi'], errors='coerce')
master_df['proximity_signal_category'] = pd.Categorical(
    np.select(
        [
            average_rssi.isna(),
            average_rssi.le(-95),
            average_rssi.gt(-95) & average_rssi.le(-85),
            average_rssi.gt(-85) & average_rssi.le(-75),
            average_rssi.gt(-75) & average_rssi.le(-60),
            average_rssi.gt(-60),
        ],
        proximity_signal_categories,
        default='no_measurement',
    ),
    categories=proximity_signal_categories,
    ordered=True,
)
master_df = master_df.drop(columns=['average_proximity_rssi'])
assert master_df['proximity_signal_category'].notna().all()

# Per-day call totals include both members' behavior with anyone. A missed call
# is an incoming call with duration -1.
max_timestamp = max(calls['timestamp'].max(), sms['timestamp'].max(), bluetooth['timestamp'].max())
total_observation_days = int(max_timestamp // SECONDS_PER_DAY) + 1

# Recompute combined daily texts received for every pair in the expanded
# universe. Values inherited from additional_pair_features only cover its
# smaller candidate-pair table, which leaves the newly created pairs blank.
combined_text_column = 'combined_daily_texts_received'
missing_combined_before = (
    int(master_df[combined_text_column].isna().sum())
    if combined_text_column in master_df.columns else len(master_df)
)
texts_received_per_day = (
    sms.groupby('recipient').size() / total_observation_days
)
master_df[combined_text_column] = (
    master_df['user_a'].map(texts_received_per_day).fillna(0.0)
    + master_df['user_b'].map(texts_received_per_day).fillna(0.0)
)
assert master_df[combined_text_column].notna().all()
print(
    f'Corrected {missing_combined_before:,} missing '
    f'{combined_text_column} values.'
)
calls_received = calls.groupby('callee').size()
calls_sent = calls.groupby('caller').size()
missed_received = calls.loc[calls['duration'].eq(-1)].groupby('callee').size()

per_user_rates = {
    'calls_received_per_day': calls_received / total_observation_days,
    'calls_sent_per_day': calls_sent / total_observation_days,
    'missed_calls_per_day': missed_received / total_observation_days,
}

# Remove any older per-member versions supplied by an interim feature file.
individual_rate_columns = [
    f'{feature_name}_{member}'
    for feature_name in per_user_rates
    for member in ('a', 'b')
]
master_df = master_df.drop(columns=individual_rate_columns, errors='ignore')

for feature_name, rates in per_user_rates.items():
    user_a_rate = master_df['user_a'].map(rates).fillna(0.0)
    user_b_rate = master_df['user_b'].map(rates).fillna(0.0)
    master_df[f'total_{feature_name}'] = user_a_rate + user_b_rate

print(f'Created requested features for {len(master_df):,} pairs across {total_observation_days} days.')


Corrected 267,852 missing combined_daily_texts_received values.
Created requested features for 350,703 pairs across 28 days.


In [15]:
pd.set_option('display.max_columns', None)
print(master_df.head())

   user_a  user_b  in_person_contact_minutes  user_a_daily_texts_received  \
0       0       1                        NaN                          NaN   
1       0       2                        NaN                          NaN   
2       0       3                        0.0                         74.0   
3       0       4                        NaN                          NaN   
4       0       5                        0.0                         74.0   

   user_b_daily_texts_received  combined_daily_texts_received  \
0                          NaN                       2.678571   
1                          NaN                       2.642857   
2                        147.0                       7.892857   
3                          NaN                       5.107143   
4                         18.0                       3.285714   

  has_completed_call is_fb_friend  days_called  total_calls  \
0                NaN          NaN          NaN          NaN   
1                NaN

In [16]:
requested_columns = [
    'combined_daily_texts_received',
    'weekend_weekday_interaction_category',
    'text_reciprocity_category',
    'call_reciprocity_category',
    'minimum_call_contact_time_category',
    'average_call_contact_time_category',
    'max_call_contact_time_category',
    'total_facebook_friend_count',
    'facebook_data_category',
    'total_facebook_friend_count_category',
    'proximity_signal_category',
    'total_calls_received_per_day',
    'total_calls_sent_per_day',
    'total_missed_calls_per_day',
]
assert not master_df[pair_keys].duplicated().any()
assert len(master_df) == len(all_users) * (len(all_users) - 1) // 2
assert set(requested_columns).issubset(master_df.columns)
assert master_df['combined_daily_texts_received'].notna().all()
assert master_df['weekend_weekday_interaction_category'].notna().all()
assert master_df[reciprocity_category_columns].notna().all().all()
assert master_df[call_duration_category_columns].notna().all().all()

# These interaction counts/durations represent no recorded interaction when missing.
zero_fill_columns = [
    'in_person_contact_minutes',
    'days_called',
    'total_calls',
    'longest_consecutive_call_streak_days',
    'longest_consecutive_text_streak_days',
    'total_texts_sent',
    'texts_shared',
    'calls_shared',
    'days_texted',
]
missing_zero_fill_columns = [
    column for column in zero_fill_columns if column not in master_df.columns
]
if missing_zero_fill_columns:
    raise KeyError(f'Cannot fill missing values; columns not found: {missing_zero_fill_columns}')

nan_counts_before_fill = master_df[zero_fill_columns].isna().sum()
master_df[zero_fill_columns] = master_df[zero_fill_columns].fillna(0)
assert master_df[zero_fill_columns].notna().all().all()
print('NaN values replaced with 0:')
print(nan_counts_before_fill)

# Save only after the requested columns have been calculated and validated.
destination_path = output_folder / output_filename
master_df.to_csv(destination_path, index=False)

print(f"\nSuccess! Combined files aligned and saved to '{destination_path}'")
print(f"Total rows in final dataset: {len(master_df)}")
print(f"Missing combined daily texts corrected: {missing_before}")
print(
    "Remaining missing combined daily texts: "
    f"{master_df[combined_column].isna().sum()}"
)


NaN values replaced with 0:
in_person_contact_minutes               267852
days_called                             350082
total_calls                             350082
longest_consecutive_call_streak_days    267852
longest_consecutive_text_streak_days    267852
total_texts_sent                        267852
texts_shared                            350006
calls_shared                            350082
days_texted                             350006
dtype: int64

Success! Combined files aligned and saved to 'C:\Users\tangc\OneDrive\Documents\cosmos\26-the-deep-learners-analysis\FinalProject\data\processed\merged_features.csv'
Total rows in final dataset: 350703
Missing combined daily texts corrected: 0
Remaining missing combined daily texts: 0


In [17]:
column = "combined_daily_texts_received"

combined_text_view = master_df[
    [
        "user_a",
        "user_b",
        "user_a_daily_texts_received",
        "user_b_daily_texts_received",
        column,
    ]
]

print(f"Total pairs: {len(combined_text_view):,}")
print(f"Missing values: {combined_text_view[column].isna().sum():,}")

display(combined_text_view.head(100))
display(combined_text_view[column].describe())

Total pairs: 350,703
Missing values: 0


,user_a,user_b,user_a_daily_texts_received,user_b_daily_texts_received,combined_daily_texts_received
0,0,1,NaN,NaN,2.678571
1,0,2,NaN,NaN,2.642857
2,0,3,74.0,147.0,7.892857
3,0,4,NaN,NaN,5.107143
4,0,5,74.0,18.0,3.285714
...,...,...,...,...,...
95,0,96,NaN,NaN,3.500000
96,0,97,NaN,NaN,3.678571
97,0,98,NaN,NaN,3.000000
98,0,99,NaN,NaN,2.785714


count    350703.000000
mean          2.074071
std           5.228179
min           0.000000
25%           0.214286
50%           0.714286
75%           2.000000
max         107.071429
Name: combined_daily_texts_received, dtype: float64

In [18]:
columns_to_remove = ['user_a_daily_texts_received', 'user_b_daily_texts_received', 'has_completed_call', 'is_fb_friend', 'fraction_rssi_above_threshold', 'fraction_rssi_below_threshold', 'active_days', 'longest_consecutive_days', 'mutual_friends', 'proximity_measurements', 'weekend_interaction_fraction', 'weekday_interaction_fraction']
master_df = master_df.drop(columns=columns_to_remove, errors='ignore')

In [19]:
# Print each column with its index number
for index, col in enumerate(master_df.columns, start=1):
    print(f"{index}. {col}")

print(f"\nTotal number of columns: {len(master_df.columns)}")


1. user_a
2. user_b
3. in_person_contact_minutes
4. combined_daily_texts_received
5. days_called
6. total_calls
7. total_facebook_friend_count
8. longest_consecutive_call_streak_days
9. longest_consecutive_text_streak_days
10. total_texts_sent
11. texts_shared
12. calls_shared
13. days_texted
14. facebook_data_category
15. total_facebook_friend_count_category
16. weekend_weekday_interaction_category
17. text_reciprocity_category
18. call_reciprocity_category
19. minimum_call_contact_time_category
20. average_call_contact_time_category
21. max_call_contact_time_category
22. proximity_signal_category
23. total_calls_received_per_day
24. total_calls_sent_per_day
25. total_missed_calls_per_day

Total number of columns: 25


In [20]:
master_df.head()

,user_a,user_b,in_person_contact_minutes,combined_daily_texts_received,days_called,total_calls,total_facebook_friend_count,longest_consecutive_call_streak_days,longest_consecutive_text_streak_days,total_texts_sent,texts_shared,calls_shared,days_texted,facebook_data_category,total_facebook_friend_count_category,weekend_weekday_interaction_category,text_reciprocity_category,call_reciprocity_category,minimum_call_contact_time_category,average_call_contact_time_category,max_call_contact_time_category,proximity_signal_category,total_calls_received_per_day,total_calls_sent_per_day,total_missed_calls_per_day
0,0,1,0.0,2.678571,0.0,0.0,25,0.0,0.0,0.0,0.0,0.0,0.0,both_have_recorded_connections,moderate,no_interaction,no_interaction,no_interaction,no_completed_call,no_completed_call,no_completed_call,no_measurement,0.214286,0.107143,0.000000
1,0,2,0.0,2.642857,0.0,0.0,26,0.0,0.0,0.0,0.0,0.0,0.0,both_have_recorded_connections,moderate,no_interaction,no_interaction,no_interaction,no_completed_call,no_completed_call,no_completed_call,no_measurement,0.214286,0.107143,0.000000
2,0,3,0.0,7.892857,0.0,0.0,37,0.0,0.0,209.0,0.0,0.0,0.0,both_have_recorded_connections,moderate,weekday_only,no_interaction,no_interaction,no_completed_call,no_completed_call,no_completed_call,weak,0.571429,0.500000,0.035714
3,0,4,0.0,5.107143,0.0,0.0,49,0.0,0.0,0.0,0.0,0.0,0.0,both_have_recorded_connections,high,no_interaction,no_interaction,no_interaction,no_completed_call,no_completed_call,no_completed_call,no_measurement,0.928571,0.964286,0.000000
4,0,5,0.0,3.285714,0.0,0.0,45,0.0,0.0,83.0,0.0,0.0,0.0,both_have_recorded_connections,moderate,weekday_only,no_interaction,no_interaction,no_completed_call,no_completed_call,no_completed_call,weak,0.392857,0.142857,0.107143


In [21]:
print(master_df.head())

   user_a  user_b  in_person_contact_minutes  combined_daily_texts_received  \
0       0       1                        0.0                       2.678571   
1       0       2                        0.0                       2.642857   
2       0       3                        0.0                       7.892857   
3       0       4                        0.0                       5.107143   
4       0       5                        0.0                       3.285714   

   days_called  total_calls  total_facebook_friend_count  \
0          0.0          0.0                           25   
1          0.0          0.0                           26   
2          0.0          0.0                           37   
3          0.0          0.0                           49   
4          0.0          0.0                           45   

   longest_consecutive_call_streak_days  longest_consecutive_text_streak_days  \
0                                   0.0                                   0.0   
1 

In [22]:
master_df.to_csv('imputed_features.csv', index=False)
